In [1]:
%pip install requests beautifulsoup4 docling ollama
# docling-vlm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.6/522.6 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.9/283.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 103.8 MB/s eta 0:00:00
 

In [ ]:
from google.colab import userdata

userdata.get("HF_TOKEN")

In [4]:
import os

import re

import sys

import time

import random

from urllib.parse import urljoin, urlparse, parse_qs

import requests

from bs4 import BeautifulSoup


from docling.datamodel.base_models import InputFormat

from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    PictureDescriptionVlmOptions,
)

from docling.document_converter import DocumentConverter, PdfFormatOption

import ollama

In [7]:
# =====================================================================
# FIX 1: Prevent Deep Layout Recursion Crashes
# =====================================================================
# EU presentation slides contain heavily nested vector graphics and maps.
# This raises the Python execution stack depth limit from 1,000 to 10,000.
sys.setrecursionlimit(10000)

# Setup distinct directories for clean organization
DOWNLOAD_DIR = "./hpai_pdfs"
OUTPUT_DIR = "./hpai_extractions"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_URL = "https://food.ec.europa.eu/horizontal-topics/committees/paff-committees/animal-health-and-welfare/presentations_en"

# Updated to a more standard browser signature
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}


def generate_clean_filename(href, link_text):
    """
    Extracts the true human-readable filename from the EU URL parameters,
    falling back safely to a sanitized version of the link text.
    """
    parsed_url = urlparse(href)
    queries = parse_qs(parsed_url.query)

    # 1. Try to pull the explicit filename parameter from the URL query
    if "filename" in queries and queries["filename"]:
        return queries["filename"][0]

    # 2. Fallback: Clean the visual link text (e.g., "HPAI - Czechia" -> "HPAI_Czechia.pdf")
    clean_text = re.sub(r"[^a-zA-Z0-9_\-]", "_", link_text.strip())
    clean_text = re.sub(r"_+", "_", clean_text).strip(
        "_"
    )  # remove duplicate underscores
    if clean_text:
        return f"{clean_text}.pdf"

    # 3. Last resort fallback
    base = os.path.basename(parsed_url.path)
    return f"{base}.pdf" if not base.lower().endswith(".pdf") else base


## =====================================================================
## PHASE 1: Crawl & Smart-Download (Skips if already local)
## =====================================================================
print(f"Fetching updates from: {TARGET_URL}")
session = requests.Session()
response = session.get(TARGET_URL, headers=HEADERS)

if response.status_code != 200:
    print(f"Failed to access server. Status: {response.status_code}")
    exit()

soup = BeautifulSoup(response.text, "html.parser")
active_pdf_queue = []

for anchor in soup.find_all("a", href=True):
    href = anchor["href"]
    link_text = anchor.get_text(strip=True)

    # only download some pdfs
    # if len(active_pdf_queue) > 20:
    #  break

    # Identify targeted HPAI presentations
    if "document/download" in href.lower() or href.lower().endswith(".pdf"):
        if re.search(r"hpai", href, re.IGNORECASE) or re.search(
            r"hpai", link_text, re.IGNORECASE
        ):
            pdf_url = urljoin(TARGET_URL, href)
            filename = generate_clean_filename(href, link_text)
            local_pdf_path = os.path.join(DOWNLOAD_DIR, filename)

            # STATE CHECK 1: Skip network request if we already have this file locally
            if (
                os.path.exists(local_pdf_path)
                and os.path.getsize(local_pdf_path) > 5000
            ):
                print(f"  [Cached] '{filename}' already exists. Skipping download.")
                active_pdf_queue.append(local_pdf_path)
                continue

            # =====================================================================
            # FIX 2: Humanize Downloader Pacing (Evasion of Rate Limits)
            # =====================================================================
            # Adds a random delay between 2.0 and 4.5 seconds to dodge server flags
            delay = 2.0 + (random.random() * 2.5)
            print(f"  Pacing connections... Sleeping {delay:.2f}s")
            time.sleep(delay)

            print(f"  [New File] Fetching -> '{filename}'...")
            try:
                file_response = session.get(
                    pdf_url, headers=HEADERS, allow_redirects=True, timeout=15
                )
                content = file_response.content

                # Double-check HTTP status code
                if file_response.status_code != 200:
                    print(
                        f"    -> Warning: Server returned status {file_response.status_code} for {filename}"
                    )
                    continue

                # Magic Byte check to ensure the server actually sent a real PDF document
                if content.startswith(b"%PDF-"):
                    with open(local_pdf_path, "wb") as f:
                        f.write(content)
                    active_pdf_queue.append(local_pdf_path)
                else:
                    print(
                        f"    -> Warning: Server returned malformed or blocked payload (HTML/403) for {filename}"
                    )
            except Exception as e:
                print(f"    -> Network error for {filename}: {e}")

print(f"\n--- Processing Queue Ready: {len(active_pdf_queue)} files to verify ---")

Fetching updates from: https://food.ec.europa.eu/horizontal-topics/committees/paff-committees/animal-health-and-welfare/presentations_en
  Pacing connections... Sleeping 4.03s
  [New File] Fetching -> 'reg-com_ahw_20260512_pres-08.pdf'...
  Pacing connections... Sleeping 3.43s
  [New File] Fetching -> 'reg-com_ahw_202604224_pres-13.pdf'...
  Pacing connections... Sleeping 3.90s
  [New File] Fetching -> 'reg-com_ahw_20260324_pres-11.pdf'...
  Pacing connections... Sleeping 3.87s
  [New File] Fetching -> 'reg-com_ahw_20260324_pres-12.pdf'...
  Pacing connections... Sleeping 2.49s
  [New File] Fetching -> 'reg-com_ahw_20260324_pres-13.pdf'...
  Pacing connections... Sleeping 4.50s
  [New File] Fetching -> 'reg-com_ahw_20260324_pres-14.pdf'...
  Pacing connections... Sleeping 2.65s
  [New File] Fetching -> 'reg-com_ahw_20260324_pres-15.pdf'...
  Pacing connections... Sleeping 2.03s
  [New File] Fetching -> 'reg-com_ahw_20260324_pres-16.pdf'...
  Pacing connections... Sleeping 2.71s
  [New 

In [10]:
## =====================================================================
## PHASE 2: Configure Pipeline Processors
## =====================================================================
pipeline_options = PdfPipelineOptions()
pipeline_options.do_picture_description = True
pipeline_options.images_scale = 2.0
pipeline_options.generate_picture_images = True
pipeline_options.picture_description_options = PictureDescriptionVlmOptions(
    repo_id="HuggingFaceTB/SmolVLM-256M-Instruct",
    prompt=(
        "Analyze this image. If it is a plot, chart, or graph, transcribe its data points, "
        "axes, labels, and key trends into a clean Markdown table. "
        "If it is a regular picture, summarize it in one sentence."
    ),
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

SYSTEM_PROMPT = (
    "You are a precise data extraction agent. You will receive a document containing text "
    "interspersed with descriptions/tables of graphs that were embedded in the file. "
    "Your job is to parse this entire text and extract the core quantitative data into a "
    "single, structured JSON object. Respond ONLY with valid JSON."
)

## =====================================================================
## PHASE 3: Idempotent Processing (Globs directly from download folder)
## =====================================================================
# Scan the local download directory for any target PDFs
downloaded_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.lower().endswith(".pdf")]

print(
    f"\n--- Processing Queue Ready: {len(downloaded_files)} target files detected locally ---"
)

for base_name in downloaded_files:
    pdf_path = os.path.join(DOWNLOAD_DIR, base_name)
    output_json_path = os.path.join(
        OUTPUT_DIR, base_name.replace(".pdf", "_extracted.json")
    )

    # STATE CHECK 2: Skip AI pipeline execution if we already generated the JSON output before
    if os.path.exists(output_json_path):
        print(
            f"  [Complete] Extraction already exists for {base_name}. Skipping AI execution."
        )
        continue

    print(f"\nExecuting Pipeline for: {base_name}")
    try:
        # Run local VLM layout parsing
        result = converter.convert(pdf_path)
        unified_markdown = result.document.export_to_markdown()

        response = unified_markdown
        # Extract structured data via local text LLM
        # response = ollama.chat(
        #    model="llama3",
        #    messages=[
        #        {"role": "system", "content": SYSTEM_PROMPT},
        #        {"role": "user", "content": f"Extract the data from this parsed text:\n\n{unified_markdown}"}
        #    ],
        #    options={"temperature": 0.0}
        # )

        # Write out final structured results
        with open(output_json_path, "w", encoding="utf-8") as json_file:
            json_file.write(response)  # response['message']['content'])

        print(f"  -> Successfully generated {os.path.basename(output_json_path)}")

    except Exception as e:
        print(f"  -> Pipeline failed on file {base_name}: {e}")

print("\n--- Pipeline Run Completed ---")


--- Processing Queue Ready: 21 target files detected locally ---
  [Complete] Extraction already exists for reg-com_ahw_20260219_pres-12.pdf. Skipping AI execution.
  [Complete] Extraction already exists for reg-com_ahw_20260324_pres-17.pdf. Skipping AI execution.

Executing Pipeline for: reg-com_ahw_20260122_pres-14.pdf


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[INFO] 2026-05-28 17:03:27,232 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-28 17:03:27,233 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-28 17:03:27,273 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-28 17:03:27,275 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-28 17:03:27,496 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-28 17:03:27,498 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-28 17:03:27,502 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-28 17:03:27,503 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  -> Successfully generated reg-com_ahw_20260122_pres-14_extracted.json

Executing Pipeline for: reg-com_ahw_20260219_pres-09.pdf


...

  -> Successfully generated reg-com_ahw_20260324_pres-12_extracted.json

Executing Pipeline for: reg-com_ahw_20260219_pres-16.pdf
  -> Pipeline failed on file reg-com_ahw_20260219_pres-16.pdf: Pipeline StandardPdfPipeline failed

Executing Pipeline for: reg-com_ahw_20260122_pres-15.pdf


[WARNING] 2026-05-28 17:13:19,546 [RapidOCR] main.py:132: The text detection result is empty


  -> Successfully generated reg-com_ahw_20260122_pres-15_extracted.json

Executing Pipeline for: reg-com_ahw_20260219_pres-15.pdf


  -> Successfully generated reg-com_ahw_20260219_pres-15_extracted.json

Executing Pipeline for: reg-com_ahw_20260324_pres-13.pdf


  -> Successfully generated reg-com_ahw_20260324_pres-13_extracted.json

Executing Pipeline for: reg-com_ahw_20260219_pres-14.pdf
  -> Successfully generated reg-com_ahw_20260219_pres-14_extracted.json

Executing Pipeline for: reg-com_ahw_20260324_pres-15.pdf
  -> Successfully generated reg-com_ahw_20260324_pres-15_extracted.json

--- Pipeline Run Completed ---


In [13]:
!zip -r ile.zip ./hpai_extractions/*

  adding: hpai_extractions/reg-com_ahw_20260122_pres-14_extracted.json (deflated 75%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-15_extracted.json (deflated 68%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-09_extracted.json (deflated 71%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-10_extracted.json (deflated 69%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-11_extracted.json (deflated 62%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-12_extracted.json (stored 0%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-13_extracted.json (deflated 56%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-14_extracted.json (deflated 75%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-15_extracted.json (deflated 67%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-18_extracted.json (deflated 61%)
  adding: hpai_extractions/reg-com_ahw_20260324_pres-12_extracted.json (deflated 59%)
  adding: hpai_extractions/reg-com_ahw_20260324_pres-13_e